<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/lab4_without_pipieline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 - Project 1: Machine Learning with PySpark MLIB (without pipeline)


### Project Overview
The objective of this project is to implement a machine learning workflow using Apache Spark's MLIB library without using a structured 'pipeline' object. Each stage of the workflow (data ingestion, data preprocessing, feature transformation, feature assembly, model training, and performance evaluation) is executed manually and sequentially.

This project utilizes the Bank Marketing Dataset (bank-full.csv). The machine learning objective is to predict whether a client will subscribe to a term deposit (binary classification with target variable 'y': 'yes' or 'no').

In [2]:
# STEP 1: install PySpark and download the dataset

!pip install pyspark

!wget -q -O bank.csv https://raw.githubusercontent.com/selva86/datasets/master/bank-full.csv

In [3]:
# STEP 2: initialize SparkSession

from pyspark.sql import SparkSession

# initialize the SparkSession with an application name
spark = SparkSession.builder.appName("BankMarketing_Without_Pipeline").getOrCreate()

# display the version
print("Spark Version: ", spark.version)

Spark Version:  4.0.4


In [4]:
# STEP 3: data ingestion and exploration

# load the csv file into a spark dataframe using semicolon as the delimiter
df = spark.read.format("csv").option("header", "true").option("sep", ";").option("inferSchema", "true").load("bank.csv")

# print the schema to inspect column names and their inferred data types
df.printSchema()

# display the first 5 rows of the raw dataset
df.show(5)

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- emp.var.rate: double (nullable = true)
 |-- cons.price.idx: double (nullable = true)
 |-- cons.conf.idx: double (nullable = true)
 |-- euribor3m: double (nullable = true)
 |-- nr.employed: double (nullable = true)
 |-- y: string (nullable = true)

+---+---------+-------+-----------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+--------------+

In [5]:
# STEP 4: feature selection and data cleaning

# import necessary functions
from pyspark.sql.functions import col, isnull, when, count

# select the relevant subset of numeric features, categorical features, and the target column
dataset = df.select(
    col("age").cast("float"),
    col("job"),
    col("marital"),
    col("education"),
    col("duration").cast("float"),
    col("campaign").cast("float"),
    col("y"),
)

# count null values for each column to verify data cleanliness
print("Checking for null values acroess selected columns:")
dataset.select([count(when(isnull(c), c)).alias(c) for c in dataset.columns]).show()

# drop rows containing null values to ensure consistent data quality
dataset = dataset.dropna(how="any")

# display the top 5 rows after cleaning and casting
dataset.show(5)


Checking for null values acroess selected columns:
+---+---+-------+---------+--------+--------+---+
|age|job|marital|education|duration|campaign|  y|
+---+---+-------+---------+--------+--------+---+
|  0|  0|      0|        0|       0|       0|  0|
+---+---+-------+---------+--------+--------+---+

+----+---------+-------+-----------+--------+--------+---+
| age|      job|marital|  education|duration|campaign|  y|
+----+---------+-------+-----------+--------+--------+---+
|56.0|housemaid|married|   basic.4y|   261.0|     1.0| no|
|57.0| services|married|high.school|   149.0|     1.0| no|
|37.0| services|married|high.school|   226.0|     1.0| no|
|40.0|   admin.|married|   basic.6y|   151.0|     1.0| no|
|56.0| services|married|high.school|   307.0|     1.0| no|
+----+---------+-------+-----------+--------+--------+---+
only showing top 5 rows


In [6]:
# STEP 5: feature preprocessing (manual StringIndexer)

# import feature engineering transformers from MLlib
from pyspark.ml.feature import StringIndexer, VectorAssembler

# sequentially index each categorical feature into numeric values
# 'handleInvalid="keep"' ensures that unseen categories during evaluation are assigned a separate index
dataset = StringIndexer(inputCol="job", outputCol="job_index", handleInvalid="keep").fit(dataset).transform(dataset)
dataset = StringIndexer(inputCol="marital", outputCol="marital_index", handleInvalid="keep").fit(dataset).transform(dataset)
dataset = StringIndexer(inputCol="education", outputCol="education_index", handleInvalid="keep").fit(dataset).transform(dataset)

# convert the binary target column 'y' ('yes'/'no') into numerical 'label' (0.0 / 1.0)
dataset = StringIndexer(inputCol="y", outputCol="label", handleInvalid="keep").fit(dataset).transform(dataset)

# remove the original categorical string columns to keep only the numerical features
dataset = dataset.drop("job", "marital", "education", "y")

# show the resulting indexed DataFrame
dataset.show(5)

+----+--------+--------+---------+-------------+---------------+-----+
| age|duration|campaign|job_index|marital_index|education_index|label|
+----+--------+--------+---------+-------------+---------------+-----+
|56.0|   261.0|     1.0|      8.0|          0.0|            4.0|  0.0|
|57.0|   149.0|     1.0|      3.0|          0.0|            1.0|  0.0|
|37.0|   226.0|     1.0|      3.0|          0.0|            1.0|  0.0|
|40.0|   151.0|     1.0|      0.0|          0.0|            5.0|  0.0|
|56.0|   307.0|     1.0|      3.0|          0.0|            1.0|  0.0|
+----+--------+--------+---------+-------------+---------------+-----+
only showing top 5 rows


In [7]:
# STEP 6: feature vector assembly

# list of all feature column names to be combined
feature_cols = ["age", "duration", "campaign", "job_index", "marital_index", "education_index"]

# VectorAssemblet combines multiple numeric columns into a single vector column
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# transform the dataset to append the new 'features' vector column
transformed_data = assembler.transform(dataset)

# display the combined features vector alongside the target label
transformed_data.select("features", "label").show(5, truncate=False)

+----------------------------+-----+
|features                    |label|
+----------------------------+-----+
|[56.0,261.0,1.0,8.0,0.0,4.0]|0.0  |
|[57.0,149.0,1.0,3.0,0.0,1.0]|0.0  |
|[37.0,226.0,1.0,3.0,0.0,1.0]|0.0  |
|[40.0,151.0,1.0,0.0,0.0,5.0]|0.0  |
|[56.0,307.0,1.0,3.0,0.0,1.0]|0.0  |
+----------------------------+-----+
only showing top 5 rows


In [9]:
# STEP 7: train/test split

 # split the dataset randomly into 80% for model training and 20% for model evaluation using a fixed random seed ('42') for reproducibility
(training_data, test_data) = transformed_data.randomSplit([0.8, 0.2], seed=42)

# print the sample count for each split
print("Training Dataset Count: ", training_data.count())
print("Test Dataset Count: ", test_data.count())

Training Dataset Count:  32977
Test Dataset Count:  8211


In [10]:
# STEP 8: model training (Random Forest Classifier)

# import the classification algorithm
from pyspark.ml.classification import RandomForestClassifier

# initialize the Random Forest model with 20 trees and a maximum depth of 5
rf = RandomForestClassifier(labelCol="label", featuresCol="features", maxDepth=5, numTrees=20,)

# fit the estimator on the training data to produce a trained model
model = rf.fit(training_data)

# generate predicitions on both training and test sets
predictions_training = model.transform(training_data)
predictions_test = model.transform(test_data)

# display the true labels, model predicitions, and calculated probabilities for 5 test samples
predictions_test.select("label", "prediction", "probability").show(5)


+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|  0.0|       0.0|[0.93349613301772...|
|  0.0|       0.0|[0.93349613301772...|
|  0.0|       0.0|[0.93349613301772...|
|  1.0|       0.0|[0.87242894792431...|
|  1.0|       0.0|[0.70194841336322...|
+-----+----------+--------------------+
only showing top 5 rows


In [11]:
# STEP 9: model evaluation

# import the evaluation module
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# evaluator configured for accuracy metric
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
train_acc = evaluator_acc.evaluate(predictions_training)
test_acc = evaluator_acc.evaluate(predictions_test)

# evaluator configured for weighted F1-score metric
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1")
train_f1 = evaluator_f1.evaluate(predictions_training)
test_f1 = evaluator_f1.evaluate(predictions_test)

# output final evaluation metrics
print("=== Results without Pipeline ===")
print("Training Accuracy: ", train_acc)
print("Test Accuracy: ", test_acc)
print("Training F1-Score: ", train_f1)
print("Test F1-Score: ", test_f1)


=== Results without Pipeline ===
Training Accuracy:  0.8956848712739182
Test Accuracy:  0.8878333942272562
Training F1-Score:  0.8641757079763189
Test F1-Score:  0.8538721687008665
